# UAV Training -- YOLO Detection

Bu notebook, UAV tespit modelini Google Colab uzerinde egitmek icin bootstrap script'ini calistirir.

**Adimlar:**
1. Google Drive'i bagla
2. GitHub repo'yu klonla
3. Bagimliliklari kur
4. Dataset'i Drive -> yerel SSD'ye aktar (tar + pv)
5. Egitimi baslat (checkpoint varsa otomatik devam)

> Asagidaki hucrede `REPO_URL` degiskenini kendi GitHub repo adresinizle degistirin.

In [ ]:
##############################################################################
# YOLO UAV Training Bootstrap
##############################################################################

# -- Configuration ----------------------------------------------------------
REPO_URL      = "https://github.com/YOUR_USER/YOUR_REPO.git"   # <- set this
REPO_BRANCH   = "main"                                          # <- set this
DRIVE_DATASET = "/content/drive/MyDrive/AIA/datasets"
LOCAL_CACHE   = "/content/datasets_local"
DRIVE_RUNS    = "/content/drive/MyDrive/AIA/runs"
TRAIN_SCRIPT  = "uav_training/train.py"
# ---------------------------------------------------------------------------

import subprocess, sys, os, glob

def _run(cmd, check=True):
    return subprocess.run(cmd, shell=True, check=check)

def _banner(msg):
    print(f"\n{'='*60}\n  {msg}\n{'='*60}")

# === 1. Mount Google Drive ===
_banner("1/6 -- Mounting Google Drive")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

if not os.path.isdir(DRIVE_DATASET):
    raise FileNotFoundError(f"Dataset not found at {DRIVE_DATASET}.")
os.makedirs(DRIVE_RUNS, exist_ok=True)
print("  Done: Drive mounted")

# === 2. Clone or Pull Repository ===
_banner("2/6 -- Setting up repository")
REPO_DIR = "/content/repo"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    _run(f"git -C {REPO_DIR} fetch --all --quiet")
    _run(f"git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}")
else:
    _run(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
print("  Done: Repo ready")

# === 3. Install Requirements ===
_banner("3/6 -- Installing dependencies")
req_file = os.path.join(REPO_DIR, "requirements.txt")
if os.path.isfile(req_file):
    _run(f"{sys.executable} -m pip install -q -r {req_file}")
_run(f"{sys.executable} -m pip install -q ultralytics", check=False)
print("  Done: Dependencies installed")

# === 4. Dataset Streaming (Drive -> Local SSD) ===
_banner("4/6 -- Preparing dataset cache")
_run("which pv >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq pv > /dev/null 2>&1)")

if os.path.isdir(LOCAL_CACHE) and os.listdir(LOCAL_CACHE):
    print(f"  Cache exists at {LOCAL_CACHE} -- skipping copy")
else:
    os.makedirs(LOCAL_CACHE, exist_ok=True)
    du = subprocess.run(f'du -sb "{DRIVE_DATASET}" | cut -f1', shell=True, capture_output=True, text=True)
    total = du.stdout.strip()
    print(f"  Dataset size: {int(total) / (1024**3):.2f} GB")
    pv_flag = f"-s {total}" if total.isdigit() else ""
    _run(f'tar -C "{DRIVE_DATASET}" -cf - . | pv -f -pterb {pv_flag} | tar -C "{LOCAL_CACHE}" -xf -')
    print("  Done: Streaming complete")

# Symlink repo/datasets -> local cache
repo_datasets = os.path.join(REPO_DIR, "datasets")
if os.path.islink(repo_datasets) or os.path.isdir(repo_datasets):
    _run(f'rm -rf "{repo_datasets}"')
os.symlink(LOCAL_CACHE, repo_datasets)
print(f"  Symlinked {repo_datasets} -> {LOCAL_CACHE}")
_run('df -h /content | tail -1')

# === 5. Configure Output -> Drive ===
_banner("5/6 -- Configuring output")
_run(f'yolo settings runs_dir="{DRIVE_RUNS}"', check=False)
os.environ["UAV_PROJECT_DIR"] = DRIVE_RUNS
print(f"  runs_dir -> {DRIVE_RUNS}")
print(f"  UAV_PROJECT_DIR -> {DRIVE_RUNS}")

# === 6. Launch Training ===
_banner("6/6 -- Starting training")

def find_ckpt(d):
    c = glob.glob(os.path.join(d, '**', 'weights', 'last.pt'), recursive=True)
    return max(c, key=os.path.getmtime) if c else None

ckpt = find_ckpt(DRIVE_RUNS)
script = os.path.join(REPO_DIR, TRAIN_SCRIPT)

if not os.path.isfile(script):
    raise FileNotFoundError(f"Train script not found: {script}")

if ckpt:
    print(f"  Resuming from: {ckpt}")
    cmd = f'{sys.executable} "{script}" --resume'
else:
    print("  Starting fresh training")
    cmd = f'{sys.executable} "{script}"'

print(f"  Command: {cmd}")
os.chdir(REPO_DIR)
_run(cmd)
_banner("Training complete -- outputs saved to Google Drive")